<a href="https://colab.research.google.com/github/lim0119/-2025-3-2-PJ/blob/main/%ED%92%88%EC%A7%88%EC%BD%94%EB%93%9C(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from typing import List, Tuple, Dict, Any
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix,
    brier_score_loss, roc_curve, auc
)
from sklearn.calibration import calibration_curve

# --------------------------- 설정 ---------------------------
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 16,
    "report_dir": "qa_reports",
    "auc_bootstrap_iters": 1000,
    "target_specificity_levels": [0.95, 0.98], # 임상 목표 특이도
    # 그룹 편향성 분석을 위한 설정
    "gender_key": "gender",
    "age_key": "age",
    "age_groups": [(60, 70), (70, 80), (80, 100)], # 나이 그룹 (60대, 70대, 80대 이상)
}

# --------------------------- 공통 유틸리티 ---------------------------
def set_seed(seed: int = 42):
    """랜덤 시드 고정: 결과 재현성과 테스트 일관성 확보"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

# --------------------------- 데이터셋 래퍼---------------------------
class MRIDataset(torch.utils.data.Dataset):
    """
    MRI 이미지, 라벨, 환자 ID 포함 Dataset 클래스
    *참고: 실제 환경에서는 메타데이터(나이, 성별)를 포함하도록 확장되어야 합니다.
    """
    def __init__(self, items: List[Tuple[np.ndarray, int, str, Dict[str, Any]]], transforms=None):
        self.items = items
        self.transforms = transforms

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        x, y, pid, meta = self.items[idx]
        if self.transforms:
            # x = self.transforms(x)
            x = torch.tensor(x, dtype=torch.float32) # 더미 코드
        else:
            x = torch.tensor(x, dtype=torch.float32)
        return x, int(y), pid, meta

# --------------------------- 데이터셋 검사 ------------------------- ---
def dataset_checks(items: List[Tuple[np.ndarray, int, str, Dict[str, Any]]]) -> Dict[str, Any]:
    """데이터셋의 기본 통계와 잠재 문제점 확인"""
    report = {}
    shapes = [tuple(i[0].shape) for i in items]
    pids = [i[2] for i in items]
    labels = [i[1] for i in items]
    meta_list = [i[3] for i in items]

    # 메타데이터 추출
    ages = [m.get(CFG['age_key']) for m in meta_list if CFG['age_key'] in m]
    genders = [m.get(CFG['gender_key']) for m in meta_list if CFG['gender_key'] in m]

    report['데이터 개수'] = len(items)
    report['이미지 형태'] = list(set(shapes))
    report['라벨 분포'] = dict(pd.Series(labels).value_counts().to_dict())

    # 메타데이터 통계
    report['메타데이터'] = {
        '성별 분포': dict(pd.Series(genders).value_counts().to_dict()) if genders else "N/A",
        '나이 통계': {'평균': float(np.mean(ages)) if ages else np.nan,
                     '최소': float(np.min(ages)) if ages else np.nan,
                     '최대': float(np.max(ages)) if ages else np.nan}
    }

    report['경고'] = []
    if len(report['이미지 형태']) > 3:
         report['경고'].append('다양한 이미지 형태 발견 - 정규화 필요')

    return report

# --------------------------- 부트스트랩 AUC 신뢰구간 ---------------------------
def bootstrap_auc_ci(y_true: np.ndarray, y_score: np.ndarray, n_iterations: int = 1000, alpha: float = 0.95) -> Tuple[float, float]:
    """부트스트랩 방법을 사용하여 AUC의 (1-alpha) 신뢰구간을 계산합니다."""
    bootstrapped_aucs = []
    n_samples = len(y_true)
    for _ in range(n_iterations):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        if len(np.unique(y_true[indices])) < 2: continue
        try:
            score = roc_auc_score(y_true[indices], y_score[indices])
            bootstrapped_aucs.append(score)
        except ValueError:
            continue
    if not bootstrapped_aucs: return np.nan, np.nan
    sorted_aucs = np.array(sorted(bootstrapped_aucs))
    lower_bound_index = int(len(sorted_aucs) * (1 - alpha) / 2)
    upper_bound_index = int(len(sorted_aucs) * (1 - (1 - alpha) / 2))
    ci_lower = max(0.0, sorted_aucs[lower_bound_index])
    ci_upper = min(1.0, sorted_aucs[upper_bound_index])
    return float(ci_lower), float(ci_upper)


# --------------------------- 성능 평가 (AUC CI 통합) ---------------------------
def compute_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> Dict[str, Any]:
    """모델 예측 성능 계산 및 AUC 신뢰구간 포함"""
    if len(np.unique(y_true)) < 2:
        return {'AUC': np.nan, '정밀도': np.nan, '재현율': np.nan, 'F1': np.nan, '특이도': np.nan, '민감도': np.nan, 'AUC_95CI_하한': np.nan, 'AUC_95CI_상한': np.nan}

    y_pred = (y_score >= threshold).astype(int)

    # 기본 지표 계산
    auc_val = float(roc_auc_score(y_true, y_score))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec = float(recall_score(y_true, y_pred, zero_division=0))
    f1 = float(f1_score(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = (cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)) # 간략화된 예외 처리
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # AUC 신뢰구간 계산
    ci_lower, ci_upper = bootstrap_auc_ci(
        y_true,
        y_score,
        n_iterations=CFG["auc_bootstrap_iters"]
    )

    return {
        'AUC': auc_val,
        'AUC_95CI_하한': ci_lower,
        'AUC_95CI_상한': ci_upper,
        '정밀도': prec,
        '재현율': rec,
        'F1': f1,
        '특이도': specificity,
        '민감도': sensitivity,
    }

# --------------------------- ROC 분석 및 임상 지표 보고 ---------------------------
def roc_analysis_report(y_true: np.ndarray, y_score: np.ndarray, target_specs: List[float]) -> Dict[str, Any]:
    """
    특정 특이도(Specificity) 목표를 달성하는 지점의 민감도(Sensitivity) 및 임계값(Threshold) 보고
    ㄴ 임상 활용을 위한 최적 운영 지점 검증
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    report = {}

    for spec_level in target_specs:
        # specificity = 1 - fpr
        # 1 - fpr 값이 목표 특이도와 가장 가까운 지점을 찾음 (최소 오차)
        target_fpr = 1.0 - spec_level

        # 목표 FPR에 가장 가까운 인덱스 찾기
        # target_fpr보다 작거나 같은 FPR 중 가장 큰 인덱스를 선택 (FPR이 증가하는 순서)

        # 목표 FPR을 초과하지 않는 (즉, 목표 특이도보다 낮지 않은) 지점들 중
        # 목표 FPR과 가장 가까운 지점을 찾습니다.
        valid_indices = np.where(fpr <= target_fpr)[0]
        if len(valid_indices) == 0:
            # 목표 FPR보다 작은 FPR이 없는 경우 (매우 낮은 특이도)
            best_idx = 0
        else:
            # 목표 FPR에 가장 가까운 인덱스 (FPR의 오차가 최소인 지점)
            best_idx = valid_indices[np.argmin(np.abs(fpr[valid_indices] - target_fpr))]

        # 선택된 지점의 지표
        achieved_spec = 1.0 - fpr[best_idx]
        sensitivity_at_spec = tpr[best_idx]
        threshold_at_spec = thresholds[best_idx]

        key = f"특이도_{int(spec_level*100)}%"
        report[key] = {
            '목표 특이도': spec_level,
            '달성 민감도': float(sensitivity_at_spec),
            '달성 특이도': float(achieved_spec), # 실제 달성된 특이도 (근사치)
            '임계값': float(threshold_at_spec),
        }

    return report

# --------------------------- 그룹 성능 편향성 보고  ---------------------------
def group_fairness_report(items, y_score: np.ndarray) -> Dict[str, Any]:
    """
    성별 및 나이 그룹별 모델 성능(AUC, 재현율) 격차 확인
    ㄴ 공정성 및 편향성 검증
    """
    report = {}
    y_true = np.array([y for _, y, _, _ in items])
    meta_list = [meta for _, _, _, meta in items]

    # 1. 성별 편향성 검사
    gender_groups = {}
    for i, item in enumerate(items):
        gender = item[3].get(CFG['gender_key'])
        if gender in ['Male', 'Female', 'M', 'F', '남', '여']: # 실제 데이터의 문자열에 맞게 조정 필요
            group = '남성' if gender in ['Male', 'M', '남'] else '여성'
            gender_groups.setdefault(group, []).append(i)

    report['성별_성능'] = {}
    for group, indices in gender_groups.items():
        if len(np.unique(y_true[indices])) > 1:
            metrics = compute_metrics(y_true[indices], y_score[indices], threshold=0.5)
            report['성별_성능'][group] = {
                '데이터수': len(indices),
                'AUC': metrics['AUC'],
                '재현율': metrics['재현율'], # Recall은 놓치면 안 되는 환자를 얼마나 잘 잡는지 보여주므로 중요
            }
        else:
            report['성별_성능'][group] = {'데이터수': len(indices), 'AUC': np.nan, '재현율': np.nan, '경고': '단일 클래스'}

    # 2. 나이 그룹 편향성 검사
    age_groups_data = {}
    for i, item in enumerate(items):
        age = item[3].get(CFG['age_key'])
        if isinstance(age, (int, float)):
            for lower, upper in CFG['age_groups']:
                if lower <= age < upper:
                    group = f"{lower}대_{upper-1}세"
                    age_groups_data.setdefault(group, []).append(i)
                    break

    report['나이별_성능'] = {}
    for group, indices in age_groups_data.items():
        if len(np.unique(y_true[indices])) > 1:
            metrics = compute_metrics(y_true[indices], y_score[indices], threshold=0.5)
            report['나이별_성능'][group] = {
                '데이터수': len(indices),
                'AUC': metrics['AUC'],
                '재현율': metrics['재현율'],
            }
        else:
            report['나이별_성능'][group] = {'데이터수': len(indices), 'AUC': np.nan, '재현율': np.nan, '경고': '단일 클래스'}

    return report

# --------------------------- 캘리브레이션 ---------------------------
def calibration_report(y_true: np.ndarray, y_prob: np.ndarray, n_bins=10) -> Dict[str, Any]:
    """모델 확률 출력의 신뢰도 평가"""
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    brier = float(brier_score_loss(y_true, y_prob))
    return {
        'Brier 점수': brier,
        '신뢰도 곡선': {
            '실제 확률': prob_true.tolist(),
            '예측 확률': prob_pred.tolist()
        }
    }

# --------------------------- 데이터 누수 확인 ---------------------------
def patient_id_leakage_check(items_train, items_test) -> bool:
    """학습/테스트 데이터에 동일 환자가 포함되어 있는지 확인"""
    pids_train = set([pid for _, _, pid, _ in items_train])
    pids_test = set([pid for _, _, pid, _ in items_test])
    overlap = pids_train.intersection(pids_test)
    return len(overlap) == 0

# --------------------------- 리포트 생성  ---------------------------
def generate_report(path: str, report: Dict[str, Any]):
    """QA 결과를 JSON 파일로 저장"""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    # NaN 값은 JSON에 직렬화되지 않으므로, None으로 변환하는 처리가 필요할 수 있습니다.
    def default_serializer(obj):
        if isinstance(obj, float) and np.isnan(obj):
            return "NaN"
        raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

    with open(path, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False, default=default_serializer)

# --------------------------- 전체 QA 실행 예시  ---------------------------
def main_smoke_test(items, model_predict_fn):
    """
    전체 QA 테스트 실행: 데이터, 성능, 임상 지표, 공정성 검증 포함
    """
    print("---  MRI 기반 알츠하이머 진단 모델 종합 QA 시작 ---")
    report = {}

    # 1. 데이터셋 검사
    print("1. 데이터셋 기본 검사 중...")
    report['데이터셋 검사'] = dataset_checks(items)

    # 학습/테스트 임의 분할
    n = len(items)
    idx = list(range(n))
    random.shuffle(idx)
    split = int(0.7 * n)
    train_idx, test_idx = idx[:split], idx[split:]
    train = [items[i] for i in train_idx]
    test = [items[i] for i in test_idx]

    # 2. 데이터 누수 확인
    print("2. 데이터 누수 확인 중...")
    report['데이터 누수 확인'] = patient_id_leakage_check(train, test)

    # 3. 모델 성능 계산
    print("3. 모델 예측 및 성능 지표 계산 중...")
    y_true = np.array([y for _, y, _, _ in test])
    y_prob = np.array(model_predict_fn([img for img, _, _, _ in test]))

    report['기본 성능'] = compute_metrics(y_true, y_prob, threshold=0.5)

    # 4. 캘리브레이션 확인
    print("4. 캘리브레이션 (신뢰도) 분석 중...")
    report['캘리브레이션'] = calibration_report(y_true, y_prob)

    # 5. 임상 활용 지표 분석 (특정 특이도 목표 달성 여부) (신규)
    print("5. 임상 목표 지표 (특정 특이도에서의 민감도) 분석 중...")
    report['임상 목표 지표'] = roc_analysis_report(y_true, y_prob, CFG['target_specificity_levels'])

    # 6. 그룹 성능 편향성 분석 (Fairness QA) (신규)
    print("6. 그룹 성능 편향성 (성별/나이) 분석 중...")
    report['그룹 성능 편향성'] = group_fairness_report(test, y_prob)

    # 7. 리포트 저장
    report_path = os.path.join(CFG['report_dir'], 'comprehensive_qa_report.json')
    generate_report(report_path, report)
    print(f"--- ✅ 종합 QA 완료. 리포트 저장 위치: {report_path} ---")
    return report

# --------------------------- 실행 예시를 위한 더미 데이터/함수 ---------------------------
def dummy_model_predict_fn(images: List[np.ndarray]) -> List[float]:
    """더미 예측 함수: 무작위 확률 반환"""
    return [random.uniform(0.1, 0.9) for _ in images]

if __name__ == '__main__':
    # 더미 데이터 생성 (환자 ID, 3D 이미지, 라벨, 메타데이터)
    print("실행 예시를 위한 더미 데이터 생성 중...")
    N = 100
    dummy_pids = [f"P_{i:04d}" for i in range(N)]
    dummy_labels = [1 if i % 3 == 0 else 0 for i in range(N)] # 불균형한 라벨 분포 예시
    dummy_images = [np.random.rand(64, 64, 64).astype(np.float32) for _ in dummy_pids]

    # 더미 메타데이터 (공정성 검사를 위해 실제와 유사하게 구성)
    dummy_meta = []
    for i in range(N):
        gender = '남성' if i % 2 == 0 else '여성'
        age = random.randint(65, 95)
        dummy_meta.append({
            "gender": gender,
            "age": age,
            "weight": random.uniform(50, 90),
        })

    # 더미 데이터셋 항목 (이미지, 라벨, 환자 ID, 메타데이터)
    items = list(zip(dummy_images, dummy_labels, dummy_pids, dummy_meta))

    # QA 실행
    main_smoke_test(items, dummy_model_predict_fn)